# Prediccion de paradas de maquina en operacion minera

Este notebook usa una data sintetica propia para simular una operacion minera con equipos, turnos, ciclos, condicion mecanica y eventos de parada no programada.

La pregunta central es: **que equipos tienen mayor riesgo de parada y que variables explican ese riesgo?**

In [ ]:
import sys
import subprocess

def instalar_si_falta(modulo, paquete_pip=None):
    try:
        __import__(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete_pip or modulo])

instalar_si_falta("pandas")
instalar_si_falta("numpy")
instalar_si_falta("sklearn", "scikit-learn")
instalar_si_falta("matplotlib")
instalar_si_falta("seaborn")

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

## 1. Carga de datos

La data se genera con reglas operativas: mayor temperatura, vibracion, presion baja, mantenimiento vencido, turno noche y clima adverso elevan el riesgo de parada.

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "paradas_maquina_sintetico.csv"

if not DATA_PATH.exists():
    import runpy
    runpy.run_path(str(PROJECT_DIR / "src" / "generar_data_sintetica.py"), run_name="__main__")

df = pd.read_csv(DATA_PATH, parse_dates=["fecha_hora"])
df.head()

In [ ]:
print(df.shape)
df["parada_no_programada"].value_counts(normalize=True).rename("proporcion")

## 2. Exploracion operativa

En mantenimiento predictivo, antes de modelar conviene mirar donde se concentran los eventos: turno, tipo de equipo, clima, criticidad y condicion mecanica.

In [ ]:
resumen_turno = (
    df.groupby("turno")["parada_no_programada"]
    .mean()
    .mul(100)
    .reset_index(name="tasa_parada_pct")
)

plt.figure(figsize=(6, 4))
sns.barplot(data=resumen_turno, x="turno", y="tasa_parada_pct", color="#b45f06")
plt.title("Tasa de paradas por turno")
plt.ylabel("% paradas")
plt.xlabel("")
plt.show()

resumen_turno

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
variables = ["temperatura_motor_c", "vibracion_mm_s", "horas_desde_mantenimiento"]

for ax, var in zip(axes, variables):
    sns.boxplot(data=df, x="parada_no_programada", y=var, ax=ax)
    ax.set_title(var)
    ax.set_xlabel("Parada no programada")

plt.tight_layout()
plt.show()

## 3. Preparacion del modelo

Retiramos columnas que no deberian entrar al entrenamiento porque son posteriores al evento o son identificadores directos. La variable `riesgo_parada_estimado` tambien se retira porque fue usada para simular el target.

In [ ]:
target = "parada_no_programada"
columnas_excluir = [
    "fecha_hora",
    "equipo_id",
    "causa_probable",
    "riesgo_parada_estimado",
    target,
]

X = df.drop(columns=columnas_excluir)
y = df[target]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()

preproceso = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## 4. Entrenamiento

Comparamos dos modelos. Como las paradas son poco frecuentes, se usa `class_weight="balanced"` en Random Forest para dar mas importancia a la clase minoritaria.

In [ ]:
modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=220,
        max_depth=10,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=160,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

resultados = []
entrenados = {}

for nombre, modelo in modelos.items():
    pipe = Pipeline([("prep", preproceso), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    resultados.append({"modelo": nombre, "ROC_AUC": auc})
    entrenados[nombre] = pipe

pd.DataFrame(resultados).sort_values("ROC_AUC", ascending=False)

## 5. Umbral de decision

En mineria, no siempre conviene usar el umbral 0.50. Si una parada es costosa, podemos bajar el umbral para capturar mas equipos en riesgo, aceptando mas falsas alarmas.

In [ ]:
mejor_nombre = pd.DataFrame(resultados).sort_values("ROC_AUC", ascending=False).iloc[0]["modelo"]
mejor_modelo = entrenados[mejor_nombre]
prob = mejor_modelo.predict_proba(X_test)[:, 1]

umbral = 0.18
pred = (prob >= umbral).astype(int)

print("Modelo seleccionado:", mejor_nombre)
print("ROC AUC:", roc_auc_score(y_test, prob).round(3))
print(classification_report(y_test, pred, target_names=["sin parada", "parada"]))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["sin parada", "parada"], cmap="Oranges")
plt.title(f"Matriz de confusion con umbral {umbral}")
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, prob)

plt.figure(figsize=(7, 4))
plt.plot(recall, precision)
plt.title("Curva Precision-Recall")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

## 6. Importancia de variables

Esta parte traduce el modelo a una conversacion con mantenimiento: que senales estan elevando el riesgo?

In [ ]:
prep = mejor_modelo.named_steps["prep"]
modelo = mejor_modelo.named_steps["modelo"]

feature_names = prep.get_feature_names_out()
importancias = pd.DataFrame({
    "variable": feature_names,
    "importancia": modelo.feature_importances_,
}).sort_values("importancia", ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(data=importancias, y="variable", x="importancia", color="#7f6000")
plt.title("Variables mas importantes para anticipar paradas")
plt.xlabel("Importancia")
plt.ylabel("")
plt.show()

importancias

## 7. Ranking operativo de equipos

Una salida util para operaciones es priorizar equipos por riesgo promedio reciente. Esto no reemplaza al jefe de mantenimiento, pero ayuda a enfocar inspecciones.

In [ ]:
df_scoring = df.copy()
X_full = df_scoring.drop(columns=columnas_excluir)
df_scoring["prob_parada_modelo"] = mejor_modelo.predict_proba(X_full)[:, 1]

ranking = (
    df_scoring.sort_values("fecha_hora")
    .groupby(["equipo_id", "tipo_equipo", "criticidad"])
    .tail(24)
    .groupby(["equipo_id", "tipo_equipo", "criticidad"])["prob_parada_modelo"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="riesgo_promedio_ultimas_24h")
)

ranking

## 8. Lectura para portafolio

Este proyecto demuestra mineria de datos aplicada a operaciones mineras:

- convierte senales de equipos en probabilidad de parada,
- permite ajustar el umbral segun costo de falsa alarma vs parada real,
- entrega variables explicativas para mantenimiento,
- produce un ranking accionable por equipo.

La data es sintetica, pero la estructura esta disenada para conversar sobre turnos, ciclos, condicion mecanica y mantenimiento predictivo.